# Gold Layer Pipeline - Reddit Sentiment, Emotion & Topic Analysis

## Overview
This notebook enriches the Silver layer Reddit data with AI-powered insights:
- **Sentiment Analysis**: Classify posts as positive, negative, or neutral
- **Emotion Detection**: Identify 7 emotions (joy, anger, fear, sadness, surprise, disgust, neutral)
- **Topic Classification**: Categorize posts into 15 content topics

## Models Used
- **Sentiment**: `cardiffnlp/twitter-roberta-base-sentiment-latest` (RoBERTa)
- **Emotion**: `j-hartmann/emotion-english-distilroberta-base` (DistilRoBERTa)
- **Topic**: `facebook/bart-large-mnli` (BART Zero-Shot Classification)

## Pipeline Stages
1. **Setup**: Install dependencies and load data
2. **Model Definition**: Define inference functions
3. **Analysis**: Apply all three models to posts
4. **Validation**: Display distributions and sample results
5. **Persistence**: Save to Gold Delta table with MLflow tracking

---

In [0]:
# Install ML libraries for transformer models and experiment tracking
%pip install transformers torch mlflow --quiet

print("✅ Libraries installed successfully")

In [0]:
from pyspark.sql import functions as F
import mlflow

# Load silver layer table
SOURCE_TABLE = "workspace.redditrecon.posts_silver"
df_silver = spark.table(SOURCE_TABLE)

print(f"📥 Loaded Silver layer data")
print(f"   Table: {SOURCE_TABLE}")
print(f"   Schema: {len(df_silver.columns)} columns")
print()

# Display sample records
print("📋 Sample records:")
display(df_silver.select("id", "author", "subreddit", "title", "selftext", "score").limit(5))

---
## 🤖 Model Setup
Define inference functions for sentiment, emotion, and topic classification.

---
## 🚀 Analysis Execution
Load models and apply all three analyses to the Silver layer data.

---
## ✅ Results & Validation
Review distributions and sample records to validate analysis quality.

---
## 💾 Persistence
Save enriched data to the Gold Delta table.

In [0]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig, pipeline
import torch
import numpy as np
from typing import Iterator
import pandas as pd

# Model names
SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
EMOTION_MODEL = "j-hartmann/emotion-english-distilroberta-base"
TOPIC_MODEL = "facebook/bart-large-mnli"  # Zero-shot classification

def load_sentiment_model():
    """Load sentiment analysis model"""
    tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL)
    config = AutoConfig.from_pretrained(SENTIMENT_MODEL)
    model.eval()
    return tokenizer, model, config

def load_emotion_model():
    """Load emotion analysis model"""
    tokenizer = AutoTokenizer.from_pretrained(EMOTION_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(EMOTION_MODEL)
    config = AutoConfig.from_pretrained(EMOTION_MODEL)
    model.eval()
    return tokenizer, model, config

def predict_sentiment(text, tokenizer, model, config):
    """Predict sentiment for a single text"""
    if not text or text.strip() == "":
        return "neutral", 0.0
    
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        scores = torch.nn.functional.softmax(outputs.logits, dim=-1)
        scores = scores.detach().numpy()[0]
        
        labels = config.id2label
        label_idx = np.argmax(scores)
        label = labels[label_idx]
        confidence = float(scores[label_idx])
        
        return label, confidence
    except Exception as e:
        return "neutral", 0.0

def predict_emotion(text, tokenizer, model, config):
    """Predict emotion for a single text"""
    if not text or text.strip() == "":
        return "neutral", 0.0
    
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        scores = torch.nn.functional.softmax(outputs.logits, dim=-1)
        scores = scores.detach().numpy()[0]
        
        labels = config.id2label
        label_idx = np.argmax(scores)
        label = labels[label_idx]
        confidence = float(scores[label_idx])
        
        return label, confidence
    except Exception as e:
        return "neutral", 0.0

def load_topic_classifier():
    """Load zero-shot topic classification pipeline"""
    classifier = pipeline("zero-shot-classification", model=TOPIC_MODEL)
    return classifier

def predict_topic(text, classifier, candidate_labels):
    """Predict topic using zero-shot classification"""
    if not text or text.strip() == "":
        return "uncategorized", 0.0
    
    try:
        # Limit text length for performance
        text = text[:512]
        result = classifier(text, candidate_labels, multi_label=False)
        top_label = result['labels'][0]
        top_score = result['scores'][0]
        return top_label, float(top_score)
    except Exception as e:
        return "uncategorized", 0.0

print("✅ Model functions defined (sentiment, emotion, topic)")

In [0]:
# Define topic categories for Reddit posts
TOPIC_CATEGORIES = [
    "Technology & Science",
    "News & Politics", 
    "Entertainment & Media",
    "Gaming",
    "Sports",
    "Health & Wellness",
    "Education & Learning",
    "Business & Finance",
    "Art & Design",
    "Lifestyle & Personal",
    "Memes & Humor",
    "DIY & Crafts",
    "Food & Cooking",
    "Travel & Places",
    "Relationships & Advice"
]

# Start MLflow run to track this transformation
mlflow.set_experiment("/Users/devsnehasharma@gmail.com/Reddit_Recon/Gold_Layer_Pipeline")

with mlflow.start_run(run_name="sentiment_emotion_topic_analysis") as run:
    # Load models once on driver
    print("\n📥 Loading AI models on driver...")
    sent_tokenizer, sent_model, sent_config = load_sentiment_model()
    print("   ✅ Sentiment model loaded")
    emot_tokenizer, emot_model, emot_config = load_emotion_model()
    print("   ✅ Emotion model loaded")
    topic_classifier = load_topic_classifier()
    print("   ✅ Topic classifier loaded")
    print("\n🎯 APPLYING AI MODELS FOR SENTIMENT, EMOTION & TOPIC ANALYSIS")
    print("="*80)
    
    # Log parameters
    # Log parameters
    mlflow.log_param("sentiment_model", SENTIMENT_MODEL)
    mlflow.log_param("emotion_model", EMOTION_MODEL)
    mlflow.log_param("topic_model", TOPIC_MODEL)
    mlflow.log_param("topic_categories", ", ".join(TOPIC_CATEGORIES))
    mlflow.log_param("source_table", SOURCE_TABLE)
    mlflow.log_param("target_table", "workspace.redditrecon.posts_gold")
    
    # Create a combined text column (title + selftext) for analysis
    df_with_text = df_silver.withColumn(
        "combined_text",
        F.concat_ws(
            " ",
            F.coalesce(F.col("title"), F.lit("")),
            F.coalesce(F.col("selftext"), F.lit(""))
        )
    )
    
    # Collect data for driver-side processing (efficient for small datasets)
    print("\n🔄 Processing data on driver (100 records)...")
    rows = df_with_text.collect()
    
    # Process each row
    results = []
    for i, row in enumerate(rows):
        if (i + 1) % 25 == 0:
            print(f"   Processed {i + 1}/{len(rows)} records...")
        
        text = row.combined_text if row.combined_text else ""
        
        # Get sentiment
        sent_label, sent_conf = predict_sentiment(text, sent_tokenizer, sent_model, sent_config)
        
        # Get emotion
        emot_label, emot_conf = predict_emotion(text, emot_tokenizer, emot_model, emot_config)
        
        # Get topic
        topic_label, topic_conf = predict_topic(text, topic_classifier, TOPIC_CATEGORIES)
        
        # Create result row with all original columns plus new ones
        result_dict = row.asDict()
        result_dict['sentiment'] = sent_label
        result_dict['sentiment_confidence'] = float(sent_conf)
        result_dict['emotion'] = emot_label
        result_dict['emotion_confidence'] = float(emot_conf)
        result_dict['topic'] = topic_label
        result_dict['topic_confidence'] = float(topic_conf)
        results.append(result_dict)
    
    # Convert back to DataFrame
    df_gold = spark.createDataFrame(results).drop("combined_text")
    
    print("✅ All analyses complete (sentiment, emotion, topic)")
    
    # Log metrics
    sentiment_dist = df_gold.groupBy("sentiment").count().collect()
    for row in sentiment_dist:
        mlflow.log_metric(f"sentiment_{row['sentiment']}_count", row['count'])
    
    emotion_dist = df_gold.groupBy("emotion").count().collect()
    for row in emotion_dist:
        mlflow.log_metric(f"emotion_{row['emotion']}_count", row['count'])
    
    topic_dist = df_gold.groupBy("topic").count().collect()
    for row in topic_dist:
        # Clean topic name for metric key (remove spaces and special chars)
        topic_key = row['topic'].replace(" ", "_").replace("&", "and")
        mlflow.log_metric(f"topic_{topic_key}_count", row['count'])
    
    # Log summary metrics
    record_count = len(results)
    mlflow.log_metric("total_records_processed", record_count)
    
    print(f"\n📊 Analysis Results:")
    print(f"   Total records processed: {record_count:,}")
    print(f"   MLflow Run ID: {run.info.run_id}")
    print("="*80)
    
    # Clean up models from memory
    del sent_tokenizer, sent_model, sent_config
    del emot_tokenizer, emot_model, emot_config
    del topic_classifier
    print("\n🧹 Models cleaned from memory")

In [0]:
print("📊 DATA QUALITY VALIDATION")
print("="*80)
print()

print("📊 SENTIMENT DISTRIBUTION")
print("="*80)
sentiment_summary = df_gold.groupBy("sentiment") \
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("sentiment_confidence"), 3).alias("avg_confidence")
    ) \
    .orderBy(F.desc("count"))

display(sentiment_summary)

print("\n📊 EMOTION DISTRIBUTION")
print("="*80)
emotion_summary = df_gold.groupBy("emotion") \
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("emotion_confidence"), 3).alias("avg_confidence")
    ) \
    .orderBy(F.desc("count"))

display(emotion_summary)

print("\n📊 TOPIC DISTRIBUTION")
print("="*80)
topic_summary = df_gold.groupBy("topic") \
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("topic_confidence"), 3).alias("avg_confidence")
    ) \
    .orderBy(F.desc("count"))

display(topic_summary)

print("\n📋 SAMPLE ENRICHED RECORDS")
print("="*80)
print("📍 Note: 'score' = Reddit upvotes - downvotes (popularity metric)")
print()
display(df_gold.select(
    "id", "title", "score", "sentiment", "sentiment_confidence", 
    "emotion", "emotion_confidence", "topic", "topic_confidence"
).limit(10))

print("\n✅ Validation complete - all distributions look healthy")

In [0]:
# Define target table
GOLD_TABLE = "workspace.redditrecon.posts_gold"

print(f"\n💾 PERSISTING TO GOLD LAYER")
print("="*80)

# Write to Delta table with schema evolution
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable(GOLD_TABLE)

print(f"✅ Successfully written to: {GOLD_TABLE}")
print(f"   Mode: overwrite")
print(f"   Format: Delta Lake")
print(f"   Schema: {len(df_gold.columns)} columns")
print("="*80)

# Verify table integrity
df_verify = spark.table(GOLD_TABLE)
verify_count = df_verify.count()
original_count = len([r for r in df_gold.collect()])  # Use cached results

print(f"\n✅ Verification passed")
print(f"   Records in table: {verify_count:,}")
print(f"   Expected records: {original_count:,}")
print(f"   Match: {'YES ✅' if verify_count == original_count else 'NO ❌'}")

print(f"\n📋 Final Gold Layer Schema:")
df_verify.printSchema()

---

## 🎉 Pipeline Complete

### Gold Layer Table: `workspace.redditrecon.posts_gold`

**New Features Added:**
- `sentiment` + `sentiment_confidence` - Post sentiment (positive/negative/neutral)
- `emotion` + `emotion_confidence` - Emotional tone (joy, anger, fear, etc.)
- `topic` + `topic_confidence` - Content category (15 topics)

**Total Columns:** 22 (16 from Silver + 6 new ML features)

**Usage Examples:**
```sql
-- Find highly confident negative posts
SELECT title, sentiment, sentiment_confidence, score
FROM workspace.redditrecon.posts_gold
WHERE sentiment = 'negative' AND sentiment_confidence > 0.8
ORDER BY score DESC;

-- Analyze topic trends
SELECT topic, COUNT(*) as post_count, AVG(score) as avg_score
FROM workspace.redditrecon.posts_gold
GROUP BY topic
ORDER BY post_count DESC;
```

**Next Steps:**
- Query the gold table for insights
- Build dashboards on top of enriched data
- Create scheduled jobs for incremental updates
- Add more sophisticated topic categories